In [ ]:
%load_ext aiida
%aiida
import urllib.parse

import ipywidgets as ipw
import traitlets as trt
from aiida import orm
from IPython.display import clear_output

from surfaces_tools.helpers import HART_2_EV
from surfaces_tools.utils.cube_results import adsorption_cube_links_html

In [ ]:
class FragmentSeparationViewer(ipw.VBox):
    pk = trt.Int(allow_none=True)

    def __init__(self):
        self._workflow_uuid = None
        self.fragment_labels = []
        self.load_btn = ipw.Button(description="Load")
        self.load_btn.on_click(self.load)
        wrong_syntax = ipw.HTML(
            value='<i class="fa fa-times" style="color:red;font-size:2em;"></i> wrong syntax',
            layout={"visibility": "hidden"},
        )
        self.pk_widget = ipw.Text(
            value="",
            description="Workflow PK:",
            style={"description_width": "initial"},
            layout={"width": "200px"},
        )

        def try_convert(value):
            wrong_syntax.layout.visibility = "hidden"
            try:
                return int(value)
            except ValueError:
                wrong_syntax.layout.visibility = "visible"
                return None

        trt.dlink((self.pk_widget, "value"), (self, "pk"), try_convert)
        self.output = ipw.Output()
        self.cube_link = ipw.HTML()
        self.pk_widget.observe(self._clear_result, names="value")
        super().__init__(
            [
                ipw.HBox([self.pk_widget, self.load_btn, wrong_syntax]),
                self.output,
                self.cube_link,
            ]
        )

    def _clear_result(self, _=None):
        self._workflow_uuid = None
        self.fragment_labels = []
        self.cube_link.value = ""
        with self.output:
            clear_output()

    def load(self, _=None):
        self._clear_result()
        with self.output:
            if not self.pk:
                print("No correct pk provided")
                return
            try:
                node = orm.load_node(self.pk)
                link = adsorption_cube_links_html(node.uuid)
                self._workflow_uuid = node.uuid
                self.cube_link.value = link
                self.parse_workchain()
                print(self.print_results())
            except Exception as exc:
                self._workflow_uuid = None
                self.fragment_labels = []
                print(f"Adsorption results are unavailable: {exc}")

    def print_results(self):
        node = orm.load_node(self._workflow_uuid)
        energies = node.outputs.energies
        output = ""
        output += f"Adsorption energy (frozen): {energies['unrelaxed_separation_energy'] * HART_2_EV:0.3f} (eV)\n"
        output += f"Adsorption energy: {energies['separation_energy'] * HART_2_EV:0.3f} (eV)\n"
        for fragment in self.fragment_labels:
            energy = (
                energies[fragment]["unrelaxed"] - energies[fragment]["relaxed"]
            ) * HART_2_EV
            output += f'Deformation energy for the "{fragment}" fragment: {round(energy, 3)} (eV)\n'
        return output

    def parse_workchain(self):
        node = orm.load_node(self._workflow_uuid)
        self.fragment_labels = sorted(
            name for name in node.inputs.fragments if name != "all"
        )

In [ ]:
viewer = FragmentSeparationViewer()
display(viewer)

In [ ]:
# Load the URL after everything is set up.
try:
    url = urllib.parse.urlsplit(jupyter_notebook_url)
    viewer.pk_widget.value = urllib.parse.parse_qs(url.query)["pk"][0]
    viewer.load()
except Exception as exception:
    print(exception)